TEXT SUMMARIZER :
steps:
1. load dataset
2. random sampling
3. preprocessing
4. Tokenization
5. load model (T5Tokenizer)
6. fine tuning(define arguments
trainer values)
7. training model
8. save and load the model
9. testing...

In [1]:
import numpy as np
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

1. Load Dataset

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [5]:
train_data.sample(5)

,id,dialogue,summary
9554,13817595,"Anna: Hi, how's life?\r\nMay: Hi, good good, y...",Anna's being sent to Bangkok for a work projec...
576,13818399,Max: How's your job hunt going?\r\nDorothy: Uh...,Dorothy has been looking for a job and sending...
9836,13865148,Siobhan: Have you seen the documentary about K...,"Siobhan proposes to watch ""Anote's Ark"", the d..."
13079,13681583,Lydia: are you back from Barcelona?\r\nRoger: ...,Roger's flight from Barcelona has just touched...
8026,13611605,"Jack: Hi\r\nMartha: Hi, what do you need???\r\...",Martha thinks Jack only talks to her when he w...


In [6]:
train_data.shape, val_data.shape

((14732, 3), (818, 3))

2. Ramdom Sampling

In [7]:
#random sampling(training only on few data).

train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [8]:
train_data.shape, val_data.shape

((4000, 3), (500, 3))

3. Data Preprossesing

In [9]:
import re

def clean_data(text):
  text = re.sub(r"\r\n", " ", text) #remove lines
  text = re.sub(r"\s+", " ", text) #remove spaces
  text = re.sub(r"<.*?>", " ", text)
  text.strip().lower()
  return text

In [10]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [11]:
train_data["dialogue"][0]

"Violet: hi! i came across this Austin's article and i thought that you might find it interesting Violet:   Claire: Hi! :) Thanks, but I've already read it. :) Claire: But thanks for thinking about me :)"

4. Tokenization

In [12]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
  inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
  target = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True) #150 cuz summary smaller than dialogue

  inputs["labels"] = target["input_ids"]
  return inputs

In [14]:
# inputs["labels"] = target["input_ids"].
# This means the model will use the dialogue tokens as input and the summary tokens as the expected output (labels) during training.

In [15]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [16]:
train_dataset[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [17]:
# input ids : token ids of dialogue
# 1 => EOS, 0 => padding

# attention mask : tells which values are valid, which values are padding vaues.

# label : token ids of summary

In [18]:
len(train_dataset[0]["input_ids"])

512

In [19]:
type(train_dataset)
type(val_dataset)

list

6. Working with our model (Load Model)

In [20]:
#NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [21]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

7. Defining Training Argumments

In [22]:
# defining training argumments

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
)

In [23]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

8. Training

In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.007881,0.379260
2,0.399775,0.367007
3,0.385724,0.362409
4,0.378581,0.361604


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2000, training_loss=0.5429902191162109, metrics={'train_runtime': 798.5361, 'train_samples_per_second': 20.037, 'train_steps_per_second': 2.505, 'total_flos': 2165468823552000.0, 'train_loss': 0.5429902191162109, 'epoch': 4.0})

In [25]:
# load model => fine-tune => save model.

Save model

In [26]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model") #we have to tokenize user input.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

load model

In [27]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model") #we have to tokenize user input.

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [28]:
def summarize_dialogue(dialogue):

  #clean
  dialogue = clean_data(dialogue)

  #tokenize
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt" #returns pytorch tensors
  ).to(device)

  #generate
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length=150,
      num_beams=4, #transformer will generate 4 different types of summaries and returns the best summary.
      early_stopping=True
  )

  # token ids convert to summary => decoding.

  #decoded our output.
  summary = tokenizer.decode(targets[0], skip_special_tokens=True) # special_tokens : SEP, EOS, PADDING
  return summary

In [29]:
test_dialogue = """
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye
"""

summary = summarize_dialogue(test_dialogue)

print("Summary : ", summary)

Summary :  Amanda has Betty's number. Lemme check Hannah's number.
